In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd

data = pd.read_csv('price_of_laptops_preprocessed.csv')

In [2]:
brand_names = data['Brand'].unique().tolist()
print(brand_names)

label_mapping = {name:i for i,name in enumerate(brand_names)}
print(label_mapping)

inv_label_mapping = {v: k for k, v in label_mapping.items()}
print(inv_label_mapping)

['apple', 'asus', 'lenovo', 'hp', 'dell', 'infinix', 'acer']
{'apple': 0, 'asus': 1, 'lenovo': 2, 'hp': 3, 'dell': 4, 'infinix': 5, 'acer': 6}
{0: 'apple', 1: 'asus', 2: 'lenovo', 3: 'hp', 4: 'dell', 5: 'infinix', 6: 'acer'}


In [3]:
data['label'] = data['Brand'].map(label_mapping)

#Data splitting into training, validation, and test sets

In [4]:
X = data[['Actual Price', 'Saving', 'label']]
y = data['Discounted Price']

In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Second split: Validation + Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Training set size:", X_train.shape)
print("Validation set size:", X_val.shape)
print("Test set size:", X_test.shape)


Training set size: (214, 3)
Validation set size: (46, 3)
Test set size: (46, 3)


In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

# Metrics
results = {
    "Dataset": ["Training", "Validation", "Test"],
    "MSE": [
        mean_squared_error(y_train, y_train_pred),
        mean_squared_error(y_val, y_val_pred),
        mean_squared_error(y_test, y_test_pred)
    ],
    "R2 Score": [
        r2_score(y_train, y_train_pred),
        r2_score(y_val, y_val_pred),
        r2_score(y_test, y_test_pred)
    ],
    "RMSE" : [
        mean_squared_error(y_train, y_train_pred)**.5,
        mean_squared_error(y_val, y_val_pred)**.5,
        mean_squared_error(y_test, y_test_pred)**.5
    ]
}

results_df = pd.DataFrame(results)
print(results_df)


      Dataset           MSE  R2 Score          RMSE
0    Training  1.569463e+08  0.993512  12527.819220
1  Validation  1.414350e+08  0.994807  11892.646184
2        Test  2.336578e+08  0.995633  15285.869091


#K Fold Cross Validation

In [8]:
from sklearn.model_selection import KFold, cross_val_score

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mse_scores = -cross_val_score(
    model, X, y, cv=kfold, scoring="neg_mean_squared_error"
)

r2_scores = cross_val_score(
    model, X, y, cv=kfold, scoring="r2"
)

print("MSE for each fold:", mse_scores)
print("Average MSE:", mse_scores.mean())

print("R2 for each fold:", r2_scores)
print("Average R2:", r2_scores.mean())


MSE for each fold: [1.27060769e+08 5.83717061e+08 1.11671677e+08 1.07100390e+08
 3.58931672e+08]
Average MSE: 257696313.9320182
R2 for each fold: [0.99465949 0.99207339 0.99243311 0.991161   0.97757669]
Average R2: 0.9895807360785043
